# Dar_028 — Environmental pipeline experiment

**Purpose:** demonstrate qualified EDA → eligibility → scoring → recommendations → inspection for Moon base-location assessment.

Keep the [terrain CNN's experiment/configuration/provenance pattern](../../notebooks/modeling/terrain_similarity/similarity_experiment.py). Use `prototype2`'s separation of hard eligibility, feature scoring and recommendations as a design reference—not its personnel rules, class hierarchy or destructive output cleanup.

This is a rule-based environmental companion, **not a trained CNN** or an operational suitability assessment. Candidate examples are from Dar_020, not the CNN's patch identities.


In [1]:
from pathlib import Path
import sys, json
sys.dont_write_bytecode = True
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'outline.md').is_file() and (p/'notebooks').is_dir())
LOCAL = ROOT/'darrel stuff'
SOURCE = LOCAL/'Pipeline_Proto'
DATA = LOCAL/'data'
sys.path.insert(0, str(SOURCE))
import pandas as pd
from IPython.display import display
import environment_engine as engine
CONFIG = SOURCE/'environment_experiment.json'
POLICY = SOURCE/'diagnostic_policy.json'
config = json.loads(CONFIG.read_text(encoding='utf-8'))
display(pd.DataFrame(config['sources'].items(), columns=['evidence_package', 'pinned_analysis_run']))


,evidence_package,pinned_analysis_run
0,candidates,processed/analysis/dar_020_candidate_environme...
1,shadowcam,processed/analysis/dar_026_shadowcam_manual/20...
2,minirf,processed/analysis/dar_027_minirf_manual/20261...


## 1. Readiness and default baseline

Verify completed input/output hashes before deriving features. Retain evidence units, support and scope. Unknown requirements are an expected result when applicable mission/equipment inputs are missing—not permission to fill gaps or skip gates.


In [2]:
default_run = engine.main(['--config', str(CONFIG)])
assert default_run.is_relative_to(DATA)
baseline = pd.read_csv(default_run/'candidate_assessments.csv')
evidence = pd.read_csv(default_run/'environment_evidence.csv')
display(evidence.groupby(['metric','status']).size().rename('candidate_examples').reset_index())
display(pd.read_csv(default_run/'hard_rule_results.csv')[['rule_id','status','reason']].drop_duplicates())
assert len(baseline) == 21 and baseline.gate_status.eq('unknown').all()
assert baseline.diagnostic_preference_score.isna().all()
assert pd.read_csv(default_run/'soft_preference_results.csv').empty
assert baseline.stage3_status.eq('unknown').all()
assert baseline.operational_preference_score.isna().all()
assert pd.read_csv(default_run/'stage3_top_k.csv').empty
display(pd.read_csv(default_run/'stage3_operational_checks.csv')[['requirement','status','reason']].drop_duplicates())
display(pd.read_csv(default_run/'stage3_evidence/audit/golden_input_coverage.csv'))


{
  "mode": "evidence_only",
  "candidates": 21,
  "pipeline_stages": [
    "verified_evidence",
    "eligibility",
    "scoring",
    "recommendations",
    "inspection"
  ],
  "gate_statuses": {
    "unknown": 21
  },
  "diagnostic_ranked_candidates": 0,
  "top_k_returned": 0,
  "operational_suitability_assessed": false,
  "preliminary_model_only": true,
  "certified_suitability": false,
  "stage3_statuses": {
    "unknown": 21
  },
  "stage3_top_k_returned": 0,
  "mission_profile_supplied": false,
  "golden_target": "outline Stage 3: mission/equipment adequacy, operating margins, failed and uncertain requirements",
  "holds": [
    "Purposive EDA components, not CNN patch identities or validated base footprints",
    "Candidate selection already used proxy solar/Earth/slope conditions: not independent validation",
    "Fixed-common phase summaries are not chronological extremes or equipment response",
    "Mean visibility is not dated power/contact continuity",
    "Image/CPR contex

,metric,status,candidate_examples
0,earth_visibility_median,observed,21
1,minirf_cpr_median,observed,19
2,minirf_cpr_median,unknown,2
3,shadowcam_radiance_median,observed,3
4,shadowcam_radiance_median,unknown,18
5,slope_q90_deg,observed,21
6,solar_visibility_median,observed,21
7,thermal_phase_median_max_K,observed,5
8,thermal_phase_median_max_K,unknown,16
9,thermal_phase_median_min_K,observed,5


,rule_id,status,reason
0,thermal_operation,unknown,missing applicable inputs/models: mission oper...
1,power_continuity,unknown,missing applicable inputs/models: dated terrai...
2,communications,unknown,missing applicable inputs/models: contact sche...
3,radiation_protection,unknown,missing applicable inputs/models: applicable e...
4,foundation_mobility,unknown,missing applicable inputs/models: local mechan...


,requirement,status,reason
0,mission_equipment_requirements_readiness,unknown,start_utc; stop_utc; crew_count; activities; e...
1,site_and_period,unknown,"Applicable requirement, mission configuration ..."
2,mission_and_crew,unknown,"Applicable requirement, mission configuration ..."
3,equipment,unknown,"Applicable requirement, mission configuration ..."
4,requirements_inventory,unknown,"Applicable requirement, mission configuration ..."
5,consumables,unknown,"Applicable requirement, mission configuration ..."
6,power,unknown,"Applicable requirement, mission configuration ..."
7,thermal,unknown,"Applicable requirement, mission configuration ..."
8,communications,unknown,"Applicable requirement, mission configuration ..."
9,protection_and_ground_hazards,unknown,"Applicable requirement, mission configuration ..."


,golden_category,datasets,notebooks,evidence_status,operational_status
0,selected_site_and_operating_period,D01 D02 D03 D04 D05 D06 D19 D20 D38,Dar_001/002/010/012/023/025/030,phase/static evidence and dated reference geom...,Unknown without applicable mission/configurati...
1,mission_duration_and_activities,M04 M07,Dar_029/031,project input absent; crew-day and time-window...,Unknown without applicable mission/configurati...
2,crew_requirements_and_consumables,M02 M10 S01 S02,Dar_029/031,published nominal load reference available; cr...,Unknown without applicable mission/configurati...
3,equipment_capabilities_and_thermal_control,M01 M04 D21 D28 R01,Dar_004/015/029/031,historical/laboratory references only; selecte...,Unknown without applicable mission/configurati...
4,power_and_reserves,D02 M07 R01,Dar_015/030/031,mean sunlight and laboratory cell response can...,Unknown without applicable mission/configurati...
5,communications,D03 D19 M08,Dar_010/030/031,reference-sphere geometry lacks terrain/relay/...,Unknown without applicable mission/configurati...
6,radiation_and_protection,D07 D08 D31 M06 S02 S03,Dar_005/029/031,scenario-qualified shielding reference; no app...,Unknown without applicable mission/configurati...
7,dust_and_physical_surface_hazards,D10 M03 D13 D14 D23 D15 D33 M05 M13,Dar_007/008/016/029/031,quality/reference findings exist; local wear/h...,Unknown without applicable mission/configurati...
8,requirements_margins_and_uncertainty,M04 M09 M14 C01 S02,Dar_022/029/031 and Pipeline_Proto,passed/failed/unknown framework; no approved c...,Unknown without applicable mission/configurati...


## 2. Explicit diagnostic experiment

The separate policy file uses unapproved proxy thresholds and equal visibility weights solely to test the pipeline. Gate first, then score passing examples, then select top-k with stable ties. These components were already selected using solar/Earth/slope proxies, so this is **not independent scientific validation**.


In [3]:
diagnostic_run = engine.main(['--config', str(CONFIG), '--policy', str(POLICY)])
assessment = pd.read_csv(diagnostic_run/'candidate_assessments.csv')
top = pd.read_csv(diagnostic_run/'diagnostic_top_k.csv')
display(assessment.groupby(['gate_status','ranking_status']).size().rename('examples').reset_index())
display(top[['candidate_id','region','diagnostic_preference_score','diagnostic_rank']])
assert assessment.gate_status.eq('pass').sum() == 14
assert assessment.gate_status.eq('fail').sum() == 7
assert len(top) == 5 and top.gate_status.eq('pass').all()


{
  "mode": "diagnostic",
  "candidates": 21,
  "pipeline_stages": [
    "verified_evidence",
    "eligibility",
    "scoring",
    "recommendations",
    "inspection"
  ],
  "gate_statuses": {
    "pass": 14,
    "fail": 7
  },
  "diagnostic_ranked_candidates": 14,
  "top_k_returned": 5,
  "operational_suitability_assessed": false,
  "preliminary_model_only": true,
  "certified_suitability": false,
  "stage3_statuses": {
    "unknown": 21
  },
  "stage3_top_k_returned": 0,
  "mission_profile_supplied": false,
  "golden_target": "outline Stage 3: mission/equipment adequacy, operating margins, failed and uncertain requirements",
  "holds": [
    "Purposive EDA components, not CNN patch identities or validated base footprints",
    "Candidate selection already used proxy solar/Earth/slope conditions: not independent validation",
    "Fixed-common phase summaries are not chronological extremes or equipment response",
    "Mean visibility is not dated power/contact continuity",
    "Image/

,gate_status,ranking_status,examples
0,fail,not_ranked,7
1,pass,diagnostic_only,14


,candidate_id,region,diagnostic_preference_score,diagnostic_rank
0,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,A3CLR22_10_Leibnitz_Beta_Plateau,0.75691,1
1,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,A3CLR22_10_Leibnitz_Beta_Plateau,0.66914,2
2,A3CLR22_2_Haworth:diagnostic_component_00014,A3CLR22_2_Haworth,0.60322,3
3,A3CLR22_4_deGerlache_Rim_2:diagnostic_componen...,A3CLR22_4_deGerlache_Rim_2,0.60215,4
4,A3CLR22_5_Connecting_Ridge_Extension:diagnosti...,A3CLR22_5_Connecting_Ridge_Extension,0.59182,5


## 3. Pipeline funnel and explanations

Like the sample's stage diagnostics, retain every stage count and per-rule result instead of presenting only a final score. Keep the full candidate manifest and source evidence alongside rankings.


In [4]:
funnel = pd.read_csv(default_run/'pipeline_stage_counts.csv').merge(
    pd.read_csv(diagnostic_run/'pipeline_stage_counts.csv'), on='stage', suffixes=('_default','_diagnostic'), validate='one_to_one')
display(funnel)
checks = pd.read_csv(diagnostic_run/'hard_rule_results.csv')
display(checks.loc[checks.status.eq('fail'), ['candidate_id','rule_id','value','operator','limit','unit','scope','reason']])
utilities = pd.read_csv(diagnostic_run/'soft_preference_results.csv')
display(utilities.loc[utilities.candidate_id.isin(top.candidate_id), ['candidate_id','preference_id','value','weight','utility']])
assert utilities.loc[utilities.candidate_id.isin(assessment.loc[assessment.gate_status.ne('pass'),'candidate_id']),'utility'].isna().all()


,stage,candidates_default,candidates_diagnostic
0,verified_candidate_input,21,21
1,hard_pass,0,14
2,hard_fail,0,7
3,hard_unknown,21,0
4,hard_unconfigured,0,0
5,complete_soft_scores,0,14
6,recommended_top_k,0,5


,candidate_id,rule_id,value,operator,limit,unit,scope,reason
5,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,illustrative_slope,8.797289,le,8.0,degree,static_map_cell_summary,Illustrative sensitivity threshold only; not a...
11,A3CLR22_11_Malapert_Massif:diagnostic_componen...,illustrative_slope,8.207219,le,8.0,degree,static_map_cell_summary,Illustrative sensitivity threshold only; not a...
14,A3CLR22_11_Malapert_Massif:diagnostic_componen...,illustrative_slope,9.149078,le,8.0,degree,static_map_cell_summary,Illustrative sensitivity threshold only; not a...
17,A3CLR22_11_Malapert_Massif:diagnostic_componen...,illustrative_slope,8.159737,le,8.0,degree,static_map_cell_summary,Illustrative sensitivity threshold only; not a...
20,A3CLR22_12_deGerlache_Kocher_Massif:diagnostic...,illustrative_slope,8.301149,le,8.0,degree,static_map_cell_summary,Illustrative sensitivity threshold only; not a...
23,A3CLR22_12_deGerlache_Kocher_Massif:diagnostic...,illustrative_slope,8.289578,le,8.0,degree,static_map_cell_summary,Illustrative sensitivity threshold only; not a...
32,A3CLR22_2_Haworth:diagnostic_component_00008,illustrative_slope,8.969322,le,8.0,degree,static_map_cell_summary,Illustrative sensitivity threshold only; not a...


,candidate_id,preference_id,value,weight,utility
0,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,sunlight_context,0.52364,1.0,0.52364
1,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,earth_context,0.81464,1.0,0.81464
4,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,sunlight_context,0.51422,1.0,0.51422
5,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,earth_context,0.99960,1.0,0.99960
18,A3CLR22_2_Haworth:diagnostic_component_00014,sunlight_context,0.53240,1.0,0.53240
19,A3CLR22_2_Haworth:diagnostic_component_00014,earth_context,0.67404,1.0,0.67404
34,A3CLR22_4_deGerlache_Rim_2:diagnostic_componen...,sunlight_context,0.58784,1.0,0.58784
35,A3CLR22_4_deGerlache_Rim_2:diagnostic_componen...,earth_context,0.61646,1.0,0.61646
38,A3CLR22_5_Connecting_Ridge_Extension:diagnosti...,sunlight_context,0.59540,1.0,0.59540
39,A3CLR22_5_Connecting_Ridge_Extension:diagnosti...,earth_context,0.58824,1.0,0.58824


## 4. Source support and integration boundary

Fixed-common phase summaries remain distinct from dated extrema/equipment response. ShadowCam/Mini-RF remain context-only. Region averages must not be silently attached to a CNN patch; connecting actual terrain candidates requires validated identity/spatial support. Reference battery/soil/radiation measurements do not supply mission-specific margins.


In [5]:
display(pd.read_csv(diagnostic_run/'image_radar_context.csv'))
for folder in (default_run, diagnostic_run):
    manifest = json.loads((folder/'provenance.json').read_text(encoding='utf-8'))
    assert manifest['status'] == 'complete'
    print(folder.relative_to(ROOT))
assert not (ROOT/'data').exists(), 'A root data directory was unexpectedly recreated'
print('All experiment outputs stay under darrel stuff/data; original terrain CNN unchanged.')


,candidate_id,metric,associated_centres,supported_centres,distinct_supported_source_pixels,median,scope,scoring_allowed
0,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,shadowcam_radiance_median,12,0,0,NaN,associated_native_centres_context_only,False
1,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,shadowcam_radiance_median,1,0,0,NaN,associated_native_centres_context_only,False
2,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,shadowcam_radiance_median,2,0,0,NaN,associated_native_centres_context_only,False
3,A3CLR22_11_Malapert_Massif:diagnostic_componen...,shadowcam_radiance_median,12,0,0,NaN,associated_native_centres_context_only,False
4,A3CLR22_11_Malapert_Massif:diagnostic_componen...,shadowcam_radiance_median,11,0,0,NaN,associated_native_centres_context_only,False
5,A3CLR22_11_Malapert_Massif:diagnostic_componen...,shadowcam_radiance_median,4,0,0,NaN,associated_native_centres_context_only,False
6,A3CLR22_12_deGerlache_Kocher_Massif:diagnostic...,shadowcam_radiance_median,3,0,0,NaN,associated_native_centres_context_only,False
7,A3CLR22_12_deGerlache_Kocher_Massif:diagnostic...,shadowcam_radiance_median,1,0,0,NaN,associated_native_centres_context_only,False
8,A3CLR22_12_deGerlache_Kocher_Massif:diagnostic...,shadowcam_radiance_median,3,0,0,NaN,associated_native_centres_context_only,False
9,A3CLR22_2_Haworth:diagnostic_component_00014,shadowcam_radiance_median,4,0,0,NaN,associated_native_centres_context_only,False


darrel stuff\data\processed\env_cnn_prototype\20261108T091834579684Z_0ee85b
darrel stuff\data\processed\env_cnn_prototype\20261108T091836015025Z_c49ed4
All experiment outputs stay under darrel stuff/data; original terrain CNN unchanged.


## Outcome and next step

The experiment verifies evidence handling and the eligibility/scoring/recommendation contract, not that these example weights select better bases. Supply applicable mission/equipment requirements and validate physical support before adopting operational checks or combining environmental preferences with terrain outputs. Run commands and feature contracts are in [README.md](README.md).
